In [1]:
# ==========================================
# 1. INSTALLATION & SETUP
# ==========================================
!pip install -q gradio faster-whisper gTTS pandas scikit-learn

import os
import re
import torch
import pandas as pd
import numpy as np
import gradio as gr
from gtts import gTTS
from faster_whisper import WhisperModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# ==========================================
# 2. DATASET CREATION (PANDAS)
# ==========================================
# Synthetic dataset covering localized agricultural and market commands in Pidgin & English
data = [
    # CHECK_MARKET_PRICE
    ("How much be bag of garri for Mile 12 market?", "CHECK_MARKET_PRICE"),
    ("Abeg wetin be the price of tomatoes in Ibadan today?", "CHECK_MARKET_PRICE"),
    ("How much them dey sell cassava for Onitsha?", "CHECK_MARKET_PRICE"),
    ("Wetin be the current price of maize per bag?", "CHECK_MARKET_PRICE"),
    ("Give me price update for bag of rice in Kano", "CHECK_MARKET_PRICE"),
    ("How much be 10kg yam for Bodija market?", "CHECK_MARKET_PRICE"),

    # PEST_ADVISORY
    ("Insects dey chop my maize leaves wetin I go spray?", "PEST_ADVISORY"),
    ("My cassava leaves turn yellow and die wetin cause am?", "PEST_ADVISORY"),
    ("Wetin be the best chemical to kill worms on tomato plant?", "PEST_ADVISORY"),
    ("Pest dey destroy my farm how I go treat am?", "PEST_ADVISORY"),
    ("How I fit protect my crops against beetle attack?", "PEST_ADVISORY"),
    ("Abeg help me my yam farm get fungus disease", "PEST_ADVISORY"),

    # ORDER_INPUTS
    ("I wan buy two bags of NPK fertilizer for my farm", "ORDER_INPUTS"),
    ("Order 5 bags of urea fertiliser make you deliver to Abeokuta", "ORDER_INPUTS"),
    ("I need high quality maize seeds for planting", "ORDER_INPUTS"),
    ("Send me pesticide for armyworm treatment", "ORDER_INPUTS"),
    ("Abeg I wan buy three bags of organic manure", "ORDER_INPUTS"),
    ("I wan order 10 bags of fertilizer for my maize farm", "ORDER_INPUTS"),

    # WEATHER_FORECAST
    ("Rain go fall for Ibadan tomorrow morning?", "WEATHER_FORECAST"),
    ("Wetin be the weather report for farming in Jos this week?", "WEATHER_FORECAST"),
    ("Abeg check if sun go heavy for Abeokuta today", "WEATHER_FORECAST"),
    ("Will it rain in Kano tomorrow for planting?", "WEATHER_FORECAST"),
    ("How weather be for farm work today in Enugu?", "WEATHER_FORECAST"),

    # CUSTOMER_SUPPORT
    ("Abeg connect me to extension officer or agricultural agent", "CUSTOMER_SUPPORT"),
    ("I need human help to inspect my farm", "CUSTOMER_SUPPORT"),
    ("Call customer support for me abeg", "CUSTOMER_SUPPORT"),
    ("My account get issue connect me to agent", "CUSTOMER_SUPPORT"),
]

# Convert to Pandas DataFrame
df = pd.DataFrame(data, columns=["utterance", "intent"])

# ==========================================
# 3. MODEL TRAINING (SCIKIT-LEARN)
# ==========================================
# Build NLP classification pipeline (TF-IDF + Logistic Regression)
model_pipeline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), lowercase=True),
    LogisticRegression(C=1.0, max_iter=200)
)

# Train the model
model_pipeline.fit(df["utterance"], df["intent"])

# ==========================================
# 4. ENTITY & SLOT EXTRACTION
# ==========================================
CROPS_AND_INPUTS = ["garri", "tomatoes", "cassava", "maize", "rice", "yam", "fertilizer", "fertiliser", "NPK", "urea", "seeds", "pesticide"]
LOCATIONS = ["Mile 12", "Ibadan", "Onitsha", "Kano", "Bodija", "Abeokuta", "Jos", "Enugu"]

def extract_slots(text):
    text_lower = text.lower()

    # Crop / Produce identification
    found_crop = "None"
    for crop in CROPS_AND_INPUTS:
        if crop.lower() in text_lower:
            found_crop = crop.title()
            break

    # Location / Market identification
    found_location = "None"
    for loc in LOCATIONS:
        if loc.lower() in text_lower:
            found_location = loc.title()
            break

    # Quantity extraction (regex for numbers/bags)
    qty_match = re.search(r'(\d+|\btwo\b|\bthree\b|\bfive\b|\bten\b)\s*(bag|bags|kg)?', text_lower)
    found_qty = qty_match.group(0).title() if qty_match else "None"

    return {
        "crop_or_input": found_crop,
        "location_or_market": found_location,
        "quantity": found_qty
    }

# ==========================================
# 5. SPEECH-TO-TEXT (FASTER-WHISPER)
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
whisper_model = WhisperModel("tiny", device=device, compute_type=compute_type)

def transcribe_audio(audio_path):
    if not audio_path:
        return ""
    segments, _ = whisper_model.transcribe(audio_path, beam_size=5)
    transcription = " ".join([segment.text for segment in segments]).strip()
    return transcription

# ==========================================
# 6. TEXT-TO-SPEECH & CORE PIPELINE
# ==========================================
CONFIDENCE_THRESHOLD = 0.55

def process_agro_command(audio_path):
    if not audio_path:
        return "No audio provided.", "UNKNOWN", "None", None

    # Step 1: Transcribe
    transcription = transcribe_audio(audio_path)
    if not transcription:
        return "Could not transcribe audio.", "UNKNOWN", "None", None

    # Step 2: Classify Intent
    probs = model_pipeline.predict_proba([transcription])[0]
    max_idx = np.argmax(probs)
    predicted_intent = model_pipeline.classes_[max_idx]
    confidence = probs[max_idx]

    # Step 3: Safety Fallback Check
    if confidence < CONFIDENCE_THRESHOLD:
        intent_display = f"UNKNOWN (Low Confidence: {confidence:.1%})"
        slots_display = "None"
        response_text = "Abeg I no properly understand wetin you talk about your farm or market query. Fit try speak am again?"
    else:
        intent_display = f"{predicted_intent} ({confidence:.1%})"
        slots = extract_slots(transcription)
        slots_display = f"Crop/Input: {slots['crop_or_input']} | Location: {slots['location_or_market']} | Qty: {slots['quantity']}"

        # Step 4: Generate Response
        if predicted_intent == "CHECK_MARKET_PRICE":
            response_text = f"The current average market price for {slots['crop_or_input']} in {slots['location_or_market']} is available. Connecting to live market data."
        elif predicted_intent == "PEST_ADVISORY":
            response_text = f"Pest alert logged for {slots['crop_or_input']}. Recommended action: Apply approved insecticide or contact an advisory officer."
        elif predicted_intent == "ORDER_INPUTS":
            response_text = f"Order received for {slots['quantity']} of {slots['crop_or_input']}. Redirecting to supplier confirmation."
        elif predicted_intent == "WEATHER_FORECAST":
            response_text = f"Checking weather report for farming in {slots['location_or_market']}."
        else:
            response_text = "Routing your request to the nearest agricultural extension officer."

    # Step 5: Synthesize TTS
    tts = gTTS(text=response_text, lang='en', tld='co.uk')
    output_audio_path = "response_output.mp3"
    tts.save(output_audio_path)

    return transcription, intent_display, slots_display, output_audio_path

# ==========================================
# 7. GRADIO WEB INTERFACE
# ==========================================
demo = gr.Interface(
    fn=process_agro_command,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath"),
    outputs=[
        gr.Textbox(label="Transcribed Pidgin Speech"),
        gr.Textbox(label="Predicted Intent & Confidence Score"),
        gr.Textbox(label="Extracted Entities (Crop, Market, Quantity)"),
        gr.Audio(label="AgroVoice Audio Response")
    ],
    title="AgroVoice: Agricultural Advisory & Market Voice Assistant",
    description="Speak or upload a farming, market price, pest, or weather command in Nigerian Pidgin or English.",
    examples=[
        ["How much be bag of garri for Mile 12 market?"],
        ["Insects dey chop my maize leaves wetin I go spray?"],
        ["I wan buy two bags of NPK fertilizer for my farm"],
        ["Rain go fall for Ibadan tomorrow morning?"]
    ],
    theme=gr.themes.Soft()
)

if __name__ == "__main__":
    demo.launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 55.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c7d6dec6cd68a94f44.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
